# Episode 11: BPE in C++: a byte-level tokenizer for a GPT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep011-bpe.ipynb)

We write a byte-level BPE tokenizer in C++, test it against a Python reference, pretrain our model on the stories in BPE, and compare the two tokenizers' models in bits per character.

[The video](https://youtu.be/_o0mJYWdKQQ) · [The episode's page](https://mvgpt.nenuadrian.com/ep011-bpe/) · [The docs](https://nenuadrian.github.io/mvGPT/bpe/) · [The code at `ep011/end`](https://github.com/nenuadrian/mvGPT/tree/ep011/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep011/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


fetch("pretrain", "pretrain-rope")

## The tests
The setup cell built the C++ tokenizer; the tests check it against the Python one.

In [ ]:
!ls bpe*.so
!python -m pytest tests/test_bpe.py -v

## Learning the merges

In [ ]:
!python data.py --bpe  # TinyStories, the BPE merges, and the stories as BPE token ids

## Pretraining with them
Ten minutes, with rotary embeddings, to `checkpoints/pretrain-rope-bpe`.

In [ ]:
!python pretrain.py 10 --rope --bpe

## C++ against Python

In [ ]:
!python bpe_bench.py

## Bits per character
Three models of ten minutes each: words, words with RoPE, and BPE.

In [ ]:
!python bits.py checkpoints/pretrain/model.pt checkpoints/pretrain-rope/model.pt checkpoints/pretrain-rope-bpe/model.pt

## A word it has never seen

In [ ]:
!python sample.py checkpoints/pretrain-rope/model.pt "Once upon a time, Zorblax the dragon" -n 1 --tokens 50 --seed 2
!python sample.py checkpoints/pretrain-rope-bpe/model.pt "Once upon a time, Zorblax the dragon" -n 1 --tokens 60 --seed 2

Next: [episode 12, Distributed training in PyTorch: 4 GPUs, DDP](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep012-distributed.ipynb)